In [8]:
import pandas as pd

for file in ["orders.csv","inventory.csv","financials.csv","vendors.csv","logistics.csv"]:
    df = pd.read_csv(file)
    print("\n", "="*60)
    print(file)
    print(df.head())
    print(df.columns.tolist())


orders.csv
       order_id  order_date  customer_id            customer_name  \
0  ORD-00109727  2021-04-24  CUST-002978            Burnett Group   
1  ORD-00023458  2022-08-01  CUST-003721  Diaz, Douglas and Smith   
2  ORD-00114532  2021-10-19  CUST-002391            Peters-Castro   
3  ORD-00027686  2023-12-23  CUST-001165            Francis-Davis   
4  ORD-00027888  2022-12-01  CUST-001939         Johnson-Williams   

               customer_country customer_segment  product_id  \
0                       Vanuatu       Mid-Market  PRD-001241   
1                         Egypt       Mid-Market  PRD-000874   
2                    Montserrat              SMB  PRD-002858   
3  United States Virgin Islands       Enterprise  PRD-000231   
4                       Romania       Government  PRD-000067   

                 product_name product_category   vendor_id  ...  \
0    Precision Component 4832        Precision  VND-000944  ...   
1   Industrial Component 2396       Industrial  VND-00

In [9]:
financials = pd.read_csv("financials.csv")

In [10]:
import pandas as pd

financials = pd.read_csv("financials.csv")
print(financials.head())

  finance_record_id record_date  fiscal_week fiscal_quarter  fiscal_year  \
0    FIN-0000000001  2021-06-06           22             Q2         2021   
1    FIN-0000000002  2022-04-14           15             Q2         2022   
2    FIN-0000000003  2024-04-10           15             Q2         2024   
3    FIN-0000000004  2021-03-07            9             Q1         2021   
4    FIN-0000000005  2024-07-18           29             Q3         2024   

       order_id vendor_id   shipment_id  transaction_type  revenue_usd  ...  \
0  ORD-00077152       NaN  SHP-00076927    Logistics Cost         0.00  ...   
1           NaN       NaN  SHP-00019919    Logistics Cost         0.00  ...   
2           NaN       NaN           NaN           Revenue      4721.80  ...   
3  ORD-00041955       NaN           NaN  Interest Expense         0.00  ...   
4           NaN       NaN           NaN           Revenue     36919.64  ...   

   inventory_value_usd  working_capital_usd  working_capital_ratio  

In [11]:
inventory_value = financials["inventory_value_usd"].sum()

accounts_receivable = financials["accounts_receivable_usd"].sum()

accounts_payable = financials["accounts_payable_usd"].sum()

working_capital = inventory_value + accounts_receivable - accounts_payable

print("Inventory Value:", inventory_value)
print("Accounts Receivable:", accounts_receivable)
print("Accounts Payable:", accounts_payable)
print("Working Capital:", working_capital)

Inventory Value: 11295205473.35
Accounts Receivable: 8398267232.4
Accounts Payable: 6259400107.099999
Working Capital: 13434072598.650002


In [13]:
import pandas as pd

inventory = pd.read_csv("inventory.csv")
financials = pd.read_csv("financials.csv")

print("Inventory Columns:")
print(inventory.columns.tolist())

print("\nFinancials Columns:")
print(financials.columns.tolist())

Inventory Columns:
['inventory_id', 'snapshot_date', 'product_id', 'product_name', 'product_category', 'sku_code', 'warehouse_id', 'warehouse_location', 'stock_on_hand', 'units_reserved', 'available_stock', 'reorder_point', 'reorder_quantity', 'safety_stock_units', 'max_stock_level', 'days_of_supply', 'vendor_id', 'vendor_lead_time_days', 'last_receipt_date', 'last_issue_date', 'stock_age_days', 'stock_classification', 'holding_cost_rate_pct', 'unit_cost_usd', 'total_inventory_value_usd', 'stockout_flag']

Financials Columns:
['finance_record_id', 'record_date', 'fiscal_week', 'fiscal_quarter', 'fiscal_year', 'order_id', 'vendor_id', 'shipment_id', 'transaction_type', 'revenue_usd', 'procurement_cost_usd', 'logistics_cost_usd', 'operational_cost_usd', 'sla_penalty_usd', 'gross_profit_usd', 'ebitda_usd', 'accounts_receivable_usd', 'accounts_payable_usd', 'inventory_value_usd', 'working_capital_usd', 'working_capital_ratio', 'cash_flow_usd', 'cash_position_usd', 'days_sales_outstanding',

In [14]:
!pip install xgboost -q

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from xgboost import XGBRegressor

# Load data
orders = pd.read_csv("orders.csv")

# Convert date
orders["order_date"] = pd.to_datetime(orders["order_date"])

# Monthly Working Capital proxy
monthly = orders.groupby(orders["order_date"].dt.to_period("M")).agg({
    "order_value_usd": "sum"
}).reset_index()

monthly["order_date"] = monthly["order_date"].astype(str)

X = np.arange(len(monthly)).reshape(-1,1)
y = monthly["order_value_usd"]

model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

model.fit(X, y)

future = np.arange(len(monthly), len(monthly)+6).reshape(-1,1)
forecast = model.predict(future)

print("Next 6 Months Working Capital Forecast")
print(forecast)

Next 6 Months Working Capital Forecast
[49295188. 49295188. 49295188. 49295188. 49295188. 49295188.]


In [16]:
import pandas as pd

orders = pd.read_csv("orders.csv")
print(orders.columns.tolist())

['order_id', 'order_date', 'customer_id', 'customer_name', 'customer_country', 'customer_segment', 'product_id', 'product_name', 'product_category', 'vendor_id', 'order_quantity', 'unit_price_usd', 'order_value_usd', 'discount_pct', 'cost_of_goods_usd', 'gross_margin_usd', 'profit_margin_pct', 'order_status', 'promised_delivery_date', 'actual_delivery_date', 'delivery_delay_days', 'fulfillment_channel', 'warehouse_id', 'shipment_id', 'return_reason', 'region', 'created_by', 'last_modified_date']


In [17]:
import pandas as pd

financials = pd.read_csv("financials.csv")
print(financials.columns.tolist())

['finance_record_id', 'record_date', 'fiscal_week', 'fiscal_quarter', 'fiscal_year', 'order_id', 'vendor_id', 'shipment_id', 'transaction_type', 'revenue_usd', 'procurement_cost_usd', 'logistics_cost_usd', 'operational_cost_usd', 'sla_penalty_usd', 'gross_profit_usd', 'ebitda_usd', 'accounts_receivable_usd', 'accounts_payable_usd', 'inventory_value_usd', 'working_capital_usd', 'working_capital_ratio', 'cash_flow_usd', 'cash_position_usd', 'days_sales_outstanding', 'days_payable_outstanding', 'cash_conversion_cycle', 'financial_risk_score', 'risk_flag']


In [18]:
inventory_value = financials["inventory_value_usd"].sum()
accounts_receivable = financials["accounts_receivable_usd"].sum()
accounts_payable = financials["accounts_payable_usd"].sum()

working_capital = inventory_value + accounts_receivable - accounts_payable

print("Inventory Value:", inventory_value)
print("Accounts Receivable:", accounts_receivable)
print("Accounts Payable:", accounts_payable)
print("Working Capital:", working_capital)

Inventory Value: 11295205473.35
Accounts Receivable: 8398267232.4
Accounts Payable: 6259400107.099999
Working Capital: 13434072598.650002
